# P2 — Reference Label Construction

## 2.1 Introduction and Objectives

This notebook constructs the encounter-level reference labels for the low back pain classification task. I use the ICD-9-CM and ICD-10-CM Uncomplicated Low Back Pain value sets from NCQA to identify encounters with recorded LBP-related diagnoses.

An encounter is labeled positive (1) if at least one recorded diagnosis matches the selected reference value sets, and negative (0) if it has valid diagnosis information but no matching code. Encounters without usable diagnosis information are not assigned a binary label.

The objectives of this notebook are to:

- Document and validate the selected reference value sets.
- Prepare source ICD codes for matching.
- Construct encounter-level binary reference labels.
- Examine label availability and class distribution.
- Prepare the reference label dataset for subsequent modeling.

These labels represent ICD-defined low back pain rather than independently validated clinical diagnoses.

## 2.2 Reference Value Set Selection

Low back pain can be represented by multiple ICD diagnosis codes. Rather than developing my own code list, I use the Uncomplicated Low Back Pain value sets from NCQA, obtained through the NLM Value Set Authority Center (VSAC). NCQA develops HEDIS, a widely used framework for healthcare quality measurement in the United States. Its value sets provide an established and reproducible ICD code reference, reducing the need to make subjective code selection decisions.

These value sets originate from the NCQA measure for the Use of Imaging Studies for Low Back Pain. I adopt the ICD code lists without implementing the full quality measure eligibility criteria. Both ICD-9-CM and ICD-10-CM are included because MIMIC-IV-ED contains diagnoses from both coding systems.

### Selected Value Sets

| Attribute | ICD-9-CM | ICD-10-CM |
|---|---|---|
| Steward | NCQA | NCQA |
| Definition Version | 20160331 | 20210925 |
| Expansion Date | 2015-04-30 | 2025-07-30 |
| Code System Version | 2013 | 2026 |
| Code Count | 24 | 83 |
| Expansion Status | Deprecated | Deprecated |

These fixed expansions are used to maintain a consistent reference definition throughout the study. They do not represent the current NCQA measure specification. Since the source data contain historical diagnoses, differences between value set versions and historical ICD coding may affect reference label coverage. The full value set OIDs and version details are documented in `docs/research_plan.md`.

**Sources:**
- [NCQA — Use of Imaging Studies for Low Back Pain](https://www.ncqa.org/report-cards/health-plans/state-of-health-care-quality-report/use-of-imaging-studies-for-low-back-pain-lbp/)
- [NLM — Value Set Authority Center](https://vsac.nlm.nih.gov/)
- [NCQA — HEDIS Measures and Technical Resources](https://www.ncqa.org/hedis/measures/)

## 2.3 Value Set Preparation and Validation

I load the original NCQA value set files and verify the code counts, completeness, uniqueness, and coding systems. The two code lists are then combined into a single reference dataset for subsequent ICD code matching.

In [14]:
from pathlib import Path

import pandas as pd
from IPython.display import display

REFERENCE_DIR = Path("../references/ncqa")
INPUT_DIR = Path("../results/01_dataset_exploration")
RESULTS_DIR = Path("../results/02_reference_labels")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load NCQA value sets
icd9_ref = pd.read_excel(
    REFERENCE_DIR / "ncqa_lbp_icd9_20150430.xlsx",
    sheet_name="Expansion List",
    skiprows=14,
    dtype=str
)

icd10_ref = pd.read_excel(
    REFERENCE_DIR / "ncqa_lbp_icd10_20250730.xlsx",
    sheet_name="Expansion List",
    skiprows=14,
    dtype=str
)

/opt/anaconda3/envs/ed-lbp-nlp/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/anaconda3/envs/ed-lbp-nlp/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [15]:
# Validate reference code lists
reference_summary = pd.DataFrame({
    "Code System": ["ICD-9-CM", "ICD-10-CM"],
    "Expected Codes": [24, 83],
    "Actual Codes": [len(icd9_ref), len(icd10_ref)],
    "Missing Codes": [
    icd9_ref["Code"].fillna("").str.strip().eq("").sum(),
    icd10_ref["Code"].fillna("").str.strip().eq("").sum()
    ],
    "Duplicate Codes": [
        icd9_ref["Code"].duplicated().sum(),
        icd10_ref["Code"].duplicated().sum()
    ]
})

display(reference_summary)

assert reference_summary["Expected Codes"].eq(
    reference_summary["Actual Codes"]
).all()

assert reference_summary["Missing Codes"].eq(0).all()
assert reference_summary["Duplicate Codes"].eq(0).all()

assert icd9_ref["Code System"].eq("ICD9CM").all()
assert icd10_ref["Code System"].eq("ICD10CM").all()

print("Reference code validation: PASSED")

,Code System,Expected Codes,Actual Codes,Missing Codes,Duplicate Codes
0,ICD-9-CM,24,24,0,0
1,ICD-10-CM,83,83,0,0


Reference code validation: PASSED


In [16]:
# Assign ICD versions
icd9_ref["icd_version"] = 9
icd10_ref["icd_version"] = 10

# Combine reference code lists
reference_codes = pd.concat(
    [icd9_ref, icd10_ref],
    ignore_index=True
)

reference_codes = reference_codes[
    ["Code", "icd_version", "Description"]
].rename(columns={
    "Code": "icd_code",
    "Description": "icd_description"
})

print(f"Total reference codes: {len(reference_codes)}")

Total reference codes: 107


## 2.4 ICD Code Preparation

The ICD codes recorded in MIMIC-IV-ED and the NCQA reference value sets may follow different formatting conventions.

Before matching, I examine the code formats and apply consistent normalization to both datasets. This step addresses formatting differences without changing the clinical meaning or scope of the selected codes. ICD versions are preserved throughout the process.

In [17]:
diagnosis_raw = pd.read_csv(
    INPUT_DIR / "diagnosis_raw.csv",
    dtype={"icd_code": str},
    keep_default_na=False
)

# Validate source data against P1
table_counts = pd.read_csv(INPUT_DIR / "table_counts.csv")

expected_count = table_counts.loc[
    table_counts["table_name"] == "diagnosis",
    "row_count"
].iloc[0]

assert len(diagnosis_raw) == expected_count, \
    "Diagnosis row count mismatch"

assert diagnosis_raw["stay_id"].notna().all()
assert diagnosis_raw["stay_id"].astype(str).str.strip().ne("").all()
assert diagnosis_raw["icd_code"].str.strip().ne("").all()
assert diagnosis_raw["icd_version"].isin([9, 10]).all()

print(f"Source diagnosis records: {len(diagnosis_raw):,}")
print("Source data validation: PASSED")

Source diagnosis records: 899,050
Source data validation: PASSED


In [18]:
# Create a unique source code inventory
source_codes = diagnosis_raw[
    ["icd_version", "icd_code"]
].drop_duplicates().reset_index(drop=True)

# Compare source and reference code formats
format_summary = pd.DataFrame({
    "Dataset": [
        "MIMIC ICD-9",
        "MIMIC ICD-10",
        "NCQA ICD-9",
        "NCQA ICD-10"
    ],
    "Unique codes": [
        len(source_codes[source_codes["icd_version"] == 9]),
        len(source_codes[source_codes["icd_version"] == 10]),
        len(icd9_ref),
        len(icd10_ref)
    ],
    "Codes with decimal point": [
        source_codes.loc[
            source_codes["icd_version"] == 9, "icd_code"
        ].str.contains(".", regex=False).sum(),

        source_codes.loc[
            source_codes["icd_version"] == 10, "icd_code"
        ].str.contains(".", regex=False).sum(),

        icd9_ref["Code"].str.contains(".", regex=False).sum(),

        icd10_ref["Code"].str.contains(".", regex=False).sum()
    ]
})

display(format_summary)

,Dataset,Unique codes,Codes with decimal point
0,MIMIC ICD-9,4684,0
1,MIMIC ICD-10,8526,0
2,NCQA ICD-9,24,24
3,NCQA ICD-10,83,83


All ICD codes in the NCQA reference value sets contain decimal points, while none of the source ICD codes do. To ensure consistent matching, I remove decimal points, trim surrounding whitespace, and convert codes to uppercase in both datasets. The original codes and ICD versions are preserved.

In [19]:
def normalize_icd(codes):
    return (
        codes.str.strip()
        .str.upper()
        .str.replace(".", "", regex=False)
    )

diagnosis_raw["normalized_code"] = normalize_icd(
    diagnosis_raw["icd_code"]
)

reference_codes["normalized_code"] = normalize_icd(
    reference_codes["icd_code"]
)

In [20]:
normalization_summary = pd.DataFrame({
    "Dataset": ["MIMIC", "NCQA"],
    "Unique codes before": [
        len(diagnosis_raw[
            ["icd_version", "icd_code"]
        ].drop_duplicates()),
        len(reference_codes[
            ["icd_version", "icd_code"]
        ].drop_duplicates())
    ],
    "Unique codes after": [
        len(diagnosis_raw[
            ["icd_version", "normalized_code"]
        ].drop_duplicates()),
        len(reference_codes[
            ["icd_version", "normalized_code"]
        ].drop_duplicates())
    ]
})

display(normalization_summary)

assert normalization_summary["Unique codes before"].eq(
    normalization_summary["Unique codes after"]
).all(), "Code normalization changed unique code counts"

print("Normalization validation: PASSED")

,Dataset,Unique codes before,Unique codes after
0,MIMIC,13210,13210
1,NCQA,107,107


Normalization validation: PASSED


## 2.5 ICD Code Matching

After normalization, I match the source ICD codes against the NCQA reference codes using both ICD version and normalized code. I first examine which reference codes are observed in MIMIC-IV-ED and then identify matching diagnosis records. Exact matching is used without expanding the reference code lists or introducing additional clinical criteria.

In [21]:
# Create a unique inventory of normalized source codes
source_code_inventory = diagnosis_raw[
    ["icd_version", "normalized_code"]
].drop_duplicates()

# Check whether each reference code exists in MIMIC
reference_coverage = reference_codes.merge(
    source_code_inventory,
    on=["icd_version", "normalized_code"],
    how="left",
    indicator=True,
    validate="one_to_one"
)

reference_coverage["observed_in_mimic"] = (
    reference_coverage["_merge"] == "both"
)

# Summarize reference code coverage
coverage_summary = (
    reference_coverage.groupby("icd_version")
    .agg(
        reference_codes=("normalized_code", "count"),
        observed_codes=("observed_in_mimic", "sum")
    )
    .reset_index()
)

coverage_summary["unobserved_codes"] = (
    coverage_summary["reference_codes"]
    - coverage_summary["observed_codes"]
)

coverage_summary["coverage_pct"] = (
    coverage_summary["observed_codes"]
    / coverage_summary["reference_codes"] * 100
).round(2)

display(coverage_summary)

,icd_version,reference_codes,observed_codes,unobserved_codes,coverage_pct
0,9,24,13,11,54.17
1,10,83,33,50,39.76


Of the 24 ICD-9-CM and 83 ICD-10-CM reference codes, 13 and 33 are observed in MIMIC-IV-ED, respectively. The unobserved codes may reflect differences in historical coding practices, diagnosis frequency, and clinical settings. Since this study adopts a fixed reference definition, all reference codes are retained without modification.

This coverage reflects the presence of reference codes in the dataset rather than the proportion of LBP encounters successfully identified.

In [22]:
# Prepare reference matching keys
reference_keys = reference_codes[
    ["icd_version", "normalized_code"]
].copy()

reference_keys["is_lbp_code"] = True

# Match source diagnoses against reference codes
diagnosis_matched = diagnosis_raw.merge(
    reference_keys,
    on=["icd_version", "normalized_code"],
    how="left",
    validate="many_to_one"
)

diagnosis_matched["is_lbp_code"] = (
    diagnosis_matched["is_lbp_code"].eq(True)
)

# Summarize matching results by ICD version
matching_summary = (
    diagnosis_matched.groupby("icd_version")
    .agg(
        total_diagnoses=("is_lbp_code", "size"),
        matched_diagnoses=("is_lbp_code", "sum")
    )
    .reset_index()
)

matching_summary["unmatched_diagnoses"] = (
    matching_summary["total_diagnoses"]
    - matching_summary["matched_diagnoses"]
)

matching_summary["matched_pct"] = (
    matching_summary["matched_diagnoses"]
    / matching_summary["total_diagnoses"] * 100
).round(2)

total_row = {
    "icd_version": "Total",
    "total_diagnoses": matching_summary["total_diagnoses"].sum(),
    "matched_diagnoses": matching_summary["matched_diagnoses"].sum(),
    "unmatched_diagnoses": matching_summary["unmatched_diagnoses"].sum()
}

total_row["matched_pct"] = round(
    total_row["matched_diagnoses"]
    / total_row["total_diagnoses"] * 100, 2
)

matching_summary = pd.concat(
    [matching_summary, pd.DataFrame([total_row])],
    ignore_index=True
)


display(matching_summary)

assert len(diagnosis_matched) == len(diagnosis_raw), \
    "Diagnosis record count mismatch"

print("Diagnosis matching row count validation: PASSED")

,icd_version,total_diagnoses,matched_diagnoses,unmatched_diagnoses,matched_pct
0,9,443015,8321,434694,1.88
1,10,456035,8473,447562,1.86
2,Total,899050,16794,882256,1.87


Diagnosis matching row count validation: PASSED


## 2.6 Encounter-Level Label Construction

Since an ED encounter may contain multiple diagnosis records, I aggregate the diagnosis-level matching results by `stay_id`.

An encounter is assigned a positive label (1) if at least one recorded ICD code matches the NCQA reference code list, and a negative label (0) if none of its recorded codes match. Encounters without usable diagnosis information are not assigned a binary label.

In [23]:
# Construct encounter-level binary labels
encounter_labels = (
    diagnosis_matched.groupby("stay_id")["is_lbp_code"]
    .any()
    .astype(int)
    .reset_index(name="lbp_label")
)

# Summarize label distribution
label_summary = (
    encounter_labels["lbp_label"]
    .value_counts()
    .sort_index()
    .rename_axis("lbp_label")
    .reset_index(name="encounter_count")
)

label_summary["percentage"] = (
    label_summary["encounter_count"]
    / len(encounter_labels) * 100
).round(2)

# Add total row
total_row = {
    "lbp_label": "Total",
    "encounter_count": len(encounter_labels),
    "percentage": 100.00
}

label_summary = pd.concat(
    [label_summary, pd.DataFrame([total_row])],
    ignore_index=True
)


display(label_summary)

,lbp_label,encounter_count,percentage
0,0,407658,96.15
1,1,16331,3.85
2,Total,423989,100.00


The positive class accounts for 3.85% of labeled ED encounters,
indicating substantial class imbalance. This will be considered
in model training and performance evaluation.

## 2.7 Reference Label Validation

Before exporting the reference labels, I verify encounter-level uniqueness, label completeness, and binary label validity.

I also check consistency with the diagnosis coverage established in P1 and the diagnosis-level matching results from Section 2.5.

In [24]:
# Load encounter identifiers from P1
edstays_raw = pd.read_csv(INPUT_DIR / "edstays_raw.csv")

# Identify expected labeled and positive encounters
diagnosed_stays = set(diagnosis_raw["stay_id"])

matched_stays = set(
    diagnosis_matched.loc[
        diagnosis_matched["is_lbp_code"], "stay_id"
    ]
)

labeled_stays = set(encounter_labels["stay_id"])

positive_stays = set(
    encounter_labels.loc[
        encounter_labels["lbp_label"] == 1, "stay_id"
    ]
)

# Validate reference labels
validation_summary = pd.DataFrame({
    "Validation": [
        "No missing encounter identifiers",
        "One label per encounter",
        "No missing labels",
        "Valid binary labels",
        "Complete diagnosis encounter coverage",
        "All labeled encounters exist in edstays",
        "Positive label consistency"
    ],
    "Passed": [
        encounter_labels["stay_id"].notna().all(),
        encounter_labels["stay_id"].is_unique,
        encounter_labels["lbp_label"].notna().all(),
        encounter_labels["lbp_label"].isin([0, 1]).all(),
        labeled_stays == diagnosed_stays,
        labeled_stays.issubset(set(edstays_raw["stay_id"])),
        positive_stays == matched_stays
    ]
})

display(validation_summary)

assert validation_summary["Passed"].all(), \
    "Reference label validation failed"

print("Reference label validation: PASSED")

,Validation,Passed
0,No missing encounter identifiers,True
1,One label per encounter,True
2,No missing labels,True
3,Valid binary labels,True
4,Complete diagnosis encounter coverage,True
5,All labeled encounters exist in edstays,True
6,Positive label consistency,True


Reference label validation: PASSED


## 2.8 Data Outputs

The validated encounter-level reference labels are saved for subsequent dataset integration and experimental setup.

The output contains one row per labeled ED encounter, with `stay_id` and `lbp_label`. Encounters without usable diagnosis information are not included. The output file is stored locally and excluded from version control.

In [25]:
# Export encounter-level reference labels
output_path = RESULTS_DIR / "encounter_reference_labels.csv"

encounter_labels.to_csv(output_path, index=False)

# Verify the exported data
exported_labels = pd.read_csv(output_path)

assert exported_labels.equals(encounter_labels), \
    "Exported data does not match source labels"

print(f"Output file: {output_path.name}")
print(f"Exported encounters: {len(exported_labels):,}")
print("Output validation: PASSED")

Output file: encounter_reference_labels.csv
Exported encounters: 423,989
Output validation: PASSED


### Reference Label Summary

| Metric | Result |
|---|---:|
| NCQA reference codes | 107 |
| Reference codes observed in MIMIC | 46 |
| Matching diagnosis records | 16,794 |
| Labeled ED encounters | 423,989 |
| Positive encounters | 16,331 |
| Negative encounters | 407,658 |
| Positive class prevalence | 3.85% |
| Encounters without diagnosis records | 1,098 |

The reference labels are constructed using exact ICD code matching against the selected NCQA value sets. These labels represent recorded ICD-defined low back pain, rather than independently validated clinical diagnoses. The labeled encounters will be integrated with chief complaint
data during experimental setup.